In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import os

# ── GPU CHECK ───────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)} — {gpus}")

# ── LOAD & PREPROCESS ────────────────────────────────────────
print("\nLoading data...")
train = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv")

Y_train_raw = train["label"].values
X_train_raw = train.drop("label", axis=1).values.reshape(-1, 28, 28, 1).astype("float32") / 255.0
X_test      = test.values.reshape(-1, 28, 28, 1).astype("float32") / 255.0

X_tr, X_val, Y_tr, Y_val = train_test_split(
    X_train_raw, Y_train_raw,
    test_size=0.1, random_state=42, stratify=Y_train_raw
)
print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


# ── CUTOUT AUGMENTATION ──────────────────────────────────────
class Cutout(layers.Layer):
    def __init__(self, num_holes=1, hole_size=8, **kwargs):
        super().__init__(**kwargs)
        self.num_holes = num_holes
        self.hole_size = hole_size

    def call(self, x, training=None):
        if not training:
            return x
        h = tf.shape(x)[1]
        w = tf.shape(x)[2]

        for _ in range(self.num_holes):
            cy = tf.random.uniform((), 0, h, dtype=tf.int32)
            cx = tf.random.uniform((), 0, w, dtype=tf.int32)
            y1 = tf.clip_by_value(cy - self.hole_size // 2, 0, h)
            y2 = tf.clip_by_value(cy + self.hole_size // 2, 0, h)
            x1 = tf.clip_by_value(cx - self.hole_size // 2, 0, w)
            x2 = tf.clip_by_value(cx + self.hole_size // 2, 0, w)

            mask = tf.ones([tf.shape(x)[0], y2 - y1, x2 - x1, tf.shape(x)[3]])
            paddings = [[0, 0], [y1, h - y2], [x1, w - x2], [0, 0]]
            mask = tf.pad(mask, paddings, constant_values=0.0)
            x = x * (1.0 - mask)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({"num_holes": self.num_holes, "hole_size": self.hole_size})
        return config


# ── AUGMENTATION BLOCK ───────────────────────────────────────
def augmentation_block():
    return models.Sequential([
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomTranslation(0.10, 0.10),
        Cutout(num_holes=1, hole_size=8),
    ], name="augmentation")


# ── ARCHITECTURE A: Strided Conv CNN ─────────────────────────
def build_model_A(seed=0):
    tf.random.set_seed(seed)
    inputs = layers.Input(shape=(28, 28, 1))
    x = augmentation_block()(inputs)

    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 5, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 5, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Conv2D(128, 4, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(10, activation='softmax', dtype='float32')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# ── ARCHITECTURE B: Wide CNN with MaxPooling ──────────────────
def build_model_B(seed=0):
    tf.random.set_seed(seed)
    inputs = layers.Input(shape=(28, 28, 1))
    x = augmentation_block()(inputs)

    x = layers.Conv2D(64, 5, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 5, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Conv2D(256, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation='softmax', dtype='float32')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# ── ARCHITECTURE C: Residual Block CNN ───────────────────────
def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_model_C(seed=0):
    tf.random.set_seed(seed)
    inputs = layers.Input(shape=(28, 28, 1))
    x = augmentation_block()(inputs)

    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = residual_block(x, 32)
    x = residual_block(x, 64, stride=2)
    x = layers.Dropout(0.3)(x)
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2)
    x = layers.Dropout(0.4)(x)
    x = residual_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(10, activation='softmax', dtype='float32')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# ── COSINE ANNEALING LR ───────────────────────────────────────
def cosine_lr(epoch, total_epochs=40, lr_max=1e-3, lr_min=1e-5):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / total_epochs))


# ── PHASE 1: BASE ENSEMBLE ────────────────────────────────────
print("\n" + "="*60)
print("Phase 1: Training Base Ensemble — 15 Models, 3 Architectures")
print("="*60)

EPOCHS_P1  = 40
BATCH_SIZE = 128

ARCH_CONFIG = [
    ("A", build_model_A, 6),
    ("B", build_model_B, 5),
    ("C", build_model_C, 4),
]

base_test_probs = np.zeros((X_test.shape[0], 10))
base_val_probs  = np.zeros((X_val.shape[0], 10))
model_count = 0

for arch_name, builder, n_models in ARCH_CONFIG:
    for i in range(n_models):
        model_count += 1
        seed = model_count * 7
        print(f"\n[{model_count}/15] Arch-{arch_name} | Model {i+1}/{n_models} | seed={seed}")

        model = builder(seed=seed)

        lr_cb = callbacks.LearningRateScheduler(
            lambda ep: cosine_lr(ep, total_epochs=EPOCHS_P1), verbose=0
        )
        es_cb = callbacks.EarlyStopping(
            monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
        )

        history = model.fit(
            X_tr, Y_tr,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS_P1,
            validation_data=(X_val, Y_val),
            callbacks=[lr_cb, es_cb],
            verbose=0
        )

        best_val = max(history.history['val_accuracy'])
        print(f"  Val accuracy: {best_val:.5f}")

        base_test_probs += model.predict(X_test, batch_size=512, verbose=0)
        base_val_probs  += model.predict(X_val,  batch_size=512, verbose=0)

base_val_labels  = np.argmax(base_val_probs, axis=1)
base_test_labels = np.argmax(base_test_probs, axis=1)

val_acc_p1 = np.mean(base_val_labels == Y_val)
print(f"\nPhase 1 complete — Ensemble val accuracy: {val_acc_p1:.5f}")

sub_p1 = pd.DataFrame({
    'ImageId': np.arange(1, len(base_test_labels) + 1),
    'Label': base_test_labels
})
sub_p1.to_csv("/kaggle/working/submission_base_ensemble.csv", index=False)
print("Saved: submission_base_ensemble.csv")


# ── PHASE 2: CONFIDENCE-GATED PSEUDO LABELING ─────────────────
print("\n" + "="*60)
print("Phase 2: Filtering High-Confidence Pseudo Labels")
print("="*60)

test_probs_norm  = base_test_probs / model_count
confidence_mask  = np.max(test_probs_norm, axis=1) > 0.995
pseudo_labels    = np.argmax(test_probs_norm, axis=1)
confident_count  = confidence_mask.sum()

print(f"  Samples with confidence > 99.5%: {confident_count}/{X_test.shape[0]} "
      f"({confident_count / X_test.shape[0] * 100:.1f}%)")

X_pseudo = np.concatenate([X_tr, X_test[confidence_mask]], axis=0)
Y_pseudo = np.concatenate([Y_tr, pseudo_labels[confidence_mask]], axis=0)

print(f"  Final training set size (train + pseudo): {X_pseudo.shape[0]}")


# ── PHASE 3: RETRAIN ON PSEUDO-LABELS ────────────────────────
print("\n" + "="*60)
print("Phase 3: Retraining Ensemble on Pseudo-Label Dataset")
print("="*60)

EPOCHS_P3 = 45
final_test_probs = np.zeros((X_test.shape[0], 10))
model_count2 = 0

for arch_name, builder, n_models in ARCH_CONFIG:
    for i in range(n_models):
        model_count2 += 1
        seed = model_count2 * 13 + 100
        print(f"\n[{model_count2}/15] Pseudo-label Arch-{arch_name} | Model {i+1}/{n_models}")

        model = builder(seed=seed)

        lr_cb = callbacks.LearningRateScheduler(
            lambda ep: cosine_lr(ep, total_epochs=EPOCHS_P3, lr_max=8e-4), verbose=0
        )
        es_cb = callbacks.EarlyStopping(
            monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
        )

        history = model.fit(
            X_pseudo, Y_pseudo,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS_P3,
            validation_data=(X_val, Y_val),
            callbacks=[lr_cb, es_cb],
            verbose=0
        )

        best_val = max(history.history['val_accuracy'])
        print(f"  Val accuracy: {best_val:.5f}")

        final_test_probs += model.predict(X_test, batch_size=512, verbose=0)

pseudo_labels_final = np.argmax(final_test_probs, axis=1)

sub_pseudo = pd.DataFrame({
    'ImageId': np.arange(1, len(pseudo_labels_final) + 1),
    'Label': pseudo_labels_final
})
sub_pseudo.to_csv("/kaggle/working/submission_pseudo_ensemble.csv", index=False)
print("Saved: submission_pseudo_ensemble.csv")


# ── PHASE 4: TEST-TIME AUGMENTATION ──────────────────────────
print("\n" + "="*60)
print("Phase 4: Test-Time Augmentation")
print("="*60)

tta_augmenter = models.Sequential([
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.08, 0.08),
], name="tta_augmenter")

TTA_STEPS = 20
print(f"  Running {TTA_STEPS} TTA steps...")

tta_probs = final_test_probs * 2  # base predictions weighted higher
for step in range(TTA_STEPS):
    X_aug = tta_augmenter(X_test, training=True).numpy()
    tta_probs += model.predict(X_aug, batch_size=512, verbose=0)
    if (step + 1) % 5 == 0:
        print(f"  TTA step {step + 1}/{TTA_STEPS} done")

final_labels = np.argmax(tta_probs, axis=1)


# ── FINAL SUBMISSION ──────────────────────────────────────────
print("\n" + "="*60)
print("Saving Final Submission")
print("="*60)

sub_final = pd.DataFrame({
    'ImageId': np.arange(1, len(final_labels) + 1),
    'Label': final_labels
})
sub_final.to_csv("/kaggle/working/submission_final.csv", index=False)

diff_from_base = (final_labels != base_test_labels).sum()
diff_from_pseudo = (final_labels != pseudo_labels_final).sum()

print(f"  Changed from base ensemble:   {diff_from_base} labels")
print(f"  Changed from pseudo ensemble: {diff_from_pseudo} labels")
print("\nAll submissions saved to /kaggle/working/:")
print("  1. submission_base_ensemble.csv   — Phase 1 result (upload this first)")
print("  2. submission_pseudo_ensemble.csv — Phase 3 result")
print("  3. submission_final.csv           — Final result with TTA (best submission)")

2026-03-28 00:54:03.812677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774659244.296845      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774659244.409039      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774659245.513038      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774659245.513086      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774659245.513089      55 computation_placer.cc:177] computation placer alr

GPUs available: 2 — [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Loading data...
Train: (37800, 28, 28, 1) | Val: (4200, 28, 28, 1) | Test: (28000, 28, 28, 1)

Phase 1: Training Base Ensemble — 15 Models, 3 Architectures

[1/15] Arch-A | Model 1/6 | seed=7


I0000 00:00:1774659288.969882      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774659288.972547      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
E0000 00:00:1774659296.584046      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1774659297.805637     126 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch 19: early stopping
Restoring model weights from the end of the best epoch: 11.
  Val accuracy: 0.98405

[2/15] Arch-A | Model 2/6 | seed=14


E0000 00:00:1774659476.009668      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_3_1/dropout_3_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


...

Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 8.
  Val accuracy: 0.98833

[14/15] Pseudo-label Arch-C | Model 3/4


E0000 00:00:1774669163.044663      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_57_1/dropout_84_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


Epoch 37: early stopping
Restoring model weights from the end of the best epoch: 29.
  Val accuracy: 0.99405

[15/15] Pseudo-label Arch-C | Model 4/4


E0000 00:00:1774669985.876270      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_59_1/dropout_87_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


Epoch 30: early stopping
Restoring model weights from the end of the best epoch: 22.
  Val accuracy: 0.99143
Saved: submission_pseudo_ensemble.csv

Phase 4: Test-Time Augmentation
  Running 20 TTA steps...
  TTA step 5/20 done
  TTA step 10/20 done
  TTA step 15/20 done
  TTA step 20/20 done

Saving Final Submission
  Changed from base ensemble:   112 labels
  Changed from pseudo ensemble: 108 labels

All submissions saved to /kaggle/working/:
  1. submission_base_ensemble.csv   — Phase 1 result (upload this first)
  2. submission_pseudo_ensemble.csv — Phase 3 result
  3. submission_final.csv           — Final result with TTA (best submission)
